<a href="https://colab.research.google.com/github/vignesh-potharaj/gen-ai/blob/main/Custotomer_Support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-genai spacy tqdm
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 33.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os
import math
import numpy as np
import spacy
from google import genai
from google.colab import userdata

# Initialize spaCy with the medium English model (which includes real word vectors)
nlp = spacy.load("en_core_web_md")

# Initialize the official Gemini Client using the Colab Secret
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("Gemini Client successfully initialized!")
except Exception as e:
    print(f"Error: Ensure you have added your 'GEMINI_API_KEY' to Colab Secrets. Details: {e}")

Gemini Client successfully initialized!


In [3]:
def softmax(x):
    """Compute softmax values for each sets of scores in x (axis=-1)."""
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

def scaled_dot_product_attention(embeddings):
    """
    Computes self-attention scores and weights.
    embeddings: numpy array of shape (seq_len, embedding_dim)
    """
    # In pure self-attention without projections, Q, K, and V are identical
    Q = embeddings
    K = embeddings
    V = embeddings

    # Get dimension of keys (d_k) for scaling factor
    d_k = K.shape[1]

    # 1. Compute Raw Dot Products (Q @ K^T)
    # Shape: (seq_len, seq_len)
    scores = np.dot(Q, K.T)

    # 2. Scale the scores to prevent exploding gradients
    scaled_scores = scores / math.sqrt(d_k)

    # 3. Apply Softmax to get Attention Weights (rows sum to 1)
    attention_weights = softmax(scaled_scores)

    # 4. Multiply by Values to get context-aware output
    context_output = np.dot(attention_weights, V)

    return attention_weights, context_output

In [4]:
# 1. Input Customer Support Query
query = "My screen is cracked and I want a refund immediately"
doc = nlp(query)

# 2. Extract words and their matching spaCy embeddings
words = [token.text for token in doc]
embeddings = np.array([token.vector for token in doc])

print(f"Tokens: {words}")
print(f"Embeddings Matrix Shape: {embeddings.shape} (Words x Vector Dimensions)\n")

# 3. Calculate Self-Attention
attention_weights, context_vectors = scaled_dot_product_attention(embeddings)

# 4. Display the Attention Matrix nicely formatted
print("--- ATTENTION WEIGHTS MATRIX (%) ---")
print(f"{'Word':<12} | " + " ".join([f"{w[:6]:^7}" for w in words]))
print("-" * (15 + 8 * len(words)))
for i, word in enumerate(words):
    weights_row = " ".join([f"{val*100:6.1f}%" for val in attention_weights[i]])
    print(f"{word:<12} | {weights_row}")

Tokens: ['My', 'screen', 'is', 'cracked', 'and', 'I', 'want', 'a', 'refund', 'immediately']
Embeddings Matrix Shape: (10, 300) (Words x Vector Dimensions)

--- ATTENTION WEIGHTS MATRIX (%) ---
Word         |   My    screen    is    cracke    and      I     want      a    refund  immedi 
-----------------------------------------------------------------------------------------------
My           |   19.9%    4.6%    8.8%    6.4%   13.1%   13.4%   12.5%    8.8%    6.2%    6.4%
screen       |    4.2%   58.8%    5.4%    3.3%    4.8%    3.6%    4.1%    5.4%    5.1%    5.4%
is           |    9.3%    6.1%   22.4%    5.1%    9.5%    6.2%    6.4%   22.4%    4.9%    7.7%
cracked      |    8.4%    4.7%    6.4%   31.4%    8.2%   12.1%    8.7%    6.4%    6.6%    7.2%
and          |   12.6%    5.0%    8.7%    6.0%   29.5%    8.7%    9.0%    8.7%    5.4%    6.5%
I            |   11.2%    3.3%    4.9%    7.7%    7.5%   37.2%   14.6%    4.9%    3.8%    4.9%
want         |   11.7%    4.2%    5.7%    6.2%